In [4]:
import os
import shutil
import pandas as pd
import numpy as np
import sys

# ==================== CONFIGURATION / КОНФИГУРАЦИЯ ====================
# (EN) Base directory containing the three classifier result folders
# (BG) Основна директория, съдържаща резултатите от трите класификатора
BASE_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_report'

# (EN) Subfolder names for the three classifiers
# (BG) Имена на подпапките за трите класификатора
CLASSIFIER_DIRS = {
    'Random Forest': os.path.join(BASE_DIR, 'random_forest'),
    'SVC': os.path.join(BASE_DIR, 'svc'),
    'Maximum Likelihood': os.path.join(BASE_DIR, 'mlc')
}

# (EN) Destination folder for the best classification results
# (BG) Крайна папка за най-добрия резултат от класификацията
BEST_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\best_classifier'
os.makedirs(BEST_DIR, exist_ok=True)

# (EN) Number of fires to process (1‑16)
# (BG) Брой пожари за обработка (1‑16)
NUM_FIRES = 16

def get_fire_id(fire_num):
    """
    (EN) Return the fire ID string used in filenames (e.g., fire_01).
    (BG) Връща идентификатора на пожар, използван в имената на файловете (напр., fire_01).
    """
    return f"fire_{fire_num:02d}"

def load_accuracy_from_metrics(metrics_path):
    """
    (EN) Load accuracy value from a classification metrics CSV.
         The CSV is expected to have a row where 'class' == 'accuracy' and a column 'accuracy'.
    (BG) Зарежда стойността на точност от CSV с метрики от класификацията.
         Очаква се ред с 'class' == 'accuracy' и колона 'accuracy'.
    """
    if not os.path.exists(metrics_path):
        return None
    try:
        df = pd.read_csv(metrics_path)
        # Find the accuracy row
        acc_row = df[df['class'] == 'accuracy']
        if acc_row.empty:
            return None
        # Extract accuracy value
        acc = acc_row['accuracy'].values[0]
        return float(acc)
    except Exception as e:
        print(f"  ⚠ Error reading {metrics_path}: {e}")
        return None

def find_best_classifier_for_fire(fire_id):
    """
    (EN) For a given fire, check the metrics of all three classifiers and return
         the name (folder key) and accuracy of the best one.
    (BG) За даден пожар проверява метриките на трите класификатора и връща
         името (ключ на папката) и точността на най-добрия.
    """
    fire_accuracies = {}
    
    for clf_name, clf_dir in CLASSIFIER_DIRS.items():
        metrics_file = os.path.join(clf_dir, f"{fire_id}_classification_metrics.csv")
        acc = load_accuracy_from_metrics(metrics_file)
        if acc is not None:
            fire_accuracies[clf_name] = acc
            print(f"    {clf_name}: accuracy = {acc:.4f} (file: {metrics_file})")
        else:
            print(f"    {clf_name}: metrics not found or unreadable")
    
    if not fire_accuracies:
        print(f"  ❌ No valid metrics for fire {fire_id}")
        return None, None
    
    # Pick best
    best_clf = max(fire_accuracies, key=fire_accuracies.get)
    best_acc = fire_accuracies[best_clf]
    print(f"  🏆 Best classifier: {best_clf} with accuracy {best_acc:.4f}")
    return best_clf, best_acc

def copy_best_files(fire_id, best_clf_name):
    """
    (EN) Copy the best classification GeoTIFF, statistics CSV, confusion matrix CSV,
         and the GeoJSON vector file from the best classifier to the BEST_DIR.
    (BG) Копира най-добрия GeoTIFF от класификацията, CSV със статистика, матрица на объркване
         и GeoJSON векторния файл от най-добрия класификатор в BEST_DIR.
    """
    src_dir = CLASSIFIER_DIRS[best_clf_name]
    
    # Mapping from classifier full name to filename-safe snake_case
    clf_name_map = {
        'Random Forest': 'random_forest',
        'SVC': 'svc',
        'Maximum Likelihood': 'maximum_likelihood'
    }
    clf_snake = clf_name_map.get(best_clf_name)
    if not clf_snake:
        print(f"  ❌ Unknown classifier mapping for {best_clf_name}")
        return False
    
    # Determine file name patterns
    tif_name = f"{fire_id}_{clf_snake}_classification.tif"
    stats_csv = f"{fire_id}_classification_statistics.csv"
    cm_csv = f"{fire_id}_confusion_matrix.csv"
    geojson_name = f"{fire_id}_{clf_snake}_classification.geojson"
    
    # ----- Source file paths -----
    # Main classifier folder (TIF, stats, CM)
    src_tif = os.path.join(src_dir, tif_name)
    src_stats = os.path.join(src_dir, stats_csv)
    src_cm = os.path.join(src_dir, cm_csv)
    
    # GeoJSON vector file is inside a 'vector_exports' subfolder of the classifier folder
    # e.g. D:\...\random_forest\vector_exports\fire_01_random_forest_classification.geojson
    vector_src_dir = os.path.join(src_dir, 'vector_exports')
    src_geojson = os.path.join(vector_src_dir, geojson_name)
    
    # ----- Destination paths (unchanged names) -----
    dst_tif = os.path.join(BEST_DIR, tif_name)
    dst_stats = os.path.join(BEST_DIR, stats_csv)
    dst_cm = os.path.join(BEST_DIR, cm_csv)
    dst_geojson = os.path.join(BEST_DIR, geojson_name)
    
    copied_anything = False
    
    # Copy GeoTIFF
    if os.path.exists(src_tif):
        shutil.copy2(src_tif, dst_tif)
        print(f"    ✅ Copied GeoTIFF: {tif_name}")
        copied_anything = True
    else:
        print(f"    ⚠ GeoTIFF not found: {tif_name}")
    
    # Copy statistics CSV
    if os.path.exists(src_stats):
        shutil.copy2(src_stats, dst_stats)
        print(f"    ✅ Copied statistics: {stats_csv}")
        copied_anything = True
    else:
        print(f"    ⚠ Statistics CSV not found: {stats_csv}")
    
    # Copy confusion matrix CSV
    if os.path.exists(src_cm):
        shutil.copy2(src_cm, dst_cm)
        print(f"    ✅ Copied confusion matrix: {cm_csv}")
        copied_anything = True
    else:
        print(f"    ⚠ Confusion matrix CSV not found: {cm_csv}")
    
    # Copy GeoJSON vector file
    if os.path.exists(src_geojson):
        shutil.copy2(src_geojson, dst_geojson)
        print(f"    ✅ Copied GeoJSON vector: {geojson_name}")
        copied_anything = True
    else:
        print(f"    ⚠ GeoJSON vector file not found: {src_geojson}")
    
    return copied_anything

def main():
    """
    (EN) Main function: for each fire, find the best classifier, and copy its results.
    (BG) Основна функция: за всеки пожар намира най-добрия класификатор и копира резултатите му.
    """
    print("🚀 Starting best classifier selection and file copy")
    print(f"📁 Output directory: {BEST_DIR}")
    print("=" * 80)
    
    summary_data = []
    successful_fires = 0
    
    for fire_num in range(1, NUM_FIRES + 1):
        fire_id = get_fire_id(fire_num)
        print(f"\n🔥 Fire {fire_id}:")
        
        best_clf, best_acc = find_best_classifier_for_fire(fire_id)
        if best_clf is None:
            print(f"  ❌ Skipping fire {fire_id} (no metrics available)")
            continue
        
        copied = copy_best_files(fire_id, best_clf)
        if copied:
            successful_fires += 1
            summary_data.append({
                'Fire ID': fire_id,
                'Best Classifier': best_clf,
                'Accuracy': f"{best_acc:.4f}"
            })
    
    # Write summary CSV
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_path = os.path.join(BEST_DIR, "best_classifier_summary.csv")
        summary_df.to_csv(summary_path, index=False)
        print(f"\n💾 Summary saved: {summary_path}")
        print("\n📊 Summary of best classifiers:")
        print(summary_df.to_string(index=False))
    
    print(f"\n✅ Successfully processed {successful_fires} out of {NUM_FIRES} fires")
    print(f"📁 Files copied to {BEST_DIR}")
    print("=" * 80)

if __name__ == "__main__":
    main()

🚀 Starting best classifier selection and file copy
📁 Output directory: D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\best_classifier

🔥 Fire fire_01:
    Random Forest: accuracy = 0.9109 (file: D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\random_forest\fire_01_classification_metrics.csv)
    SVC: accuracy = 0.8416 (file: D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\svc\fire_01_classification_metrics.csv)
    Maximum Likelihood: accuracy = 0.9010 (file: D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\mlc\fire_01_classification_metrics.csv)
  🏆 Best classifier: Random Forest with accuracy 0.9109
    ✅ Copied GeoTIFF: fire_01_random_forest_classification.tif
    ✅ Copied statistics: fire_01_classification_statistics.csv
    ✅ Copied confusion matrix: fire_01_confusion_matrix.csv
    ✅ Copied GeoJSON vector: fire_01_random_forest_classification.geojson

🔥 Fire fire_02:
    Random